In [ ]:
import sqlite3
import pandas as pd
import os

ROOTDIR = "/home/ebr/projects/release-volume-sampler"
RUNDIR = "/home/ebr/projects/release-volume-sampler/generated/messina_003"
os.chdir(os.path.join(ROOTDIR, "src"))

# Connect to the database
conn = sqlite3.connect(os.path.join(RUNDIR, "volumes", "volumes.db"))

# Query all data from the volumes table
df = pd.read_sql_query("SELECT * FROM volumes;", conn)

# Display the first few rows
print(df.head())

conn.close()

In [ ]:
df.shape

In [ ]:
from rvsampler.database_handler import VolumeDatabaseHandler

In [ ]:
with VolumeDatabaseHandler(RUNDIR) as volume_db:
    volume_db.plot_distribution()

In [ ]:
with VolumeDatabaseHandler(RUNDIR) as volume_db: 
    volume_db.plot_release_density_plots(seed_prob="p_shake")

In [ ]:
from scipy.interpolate import interp1d
import numpy as np

displacement_threshold=5.
table_filename="exceedance_displacement.npz"
column_name = "p_shake"


volume_db = VolumeDatabaseHandler(rundir)
volume_db.__enter__()


In [ ]:
COLUMN_NAMES = ['id', 'released', 'condprob',
           'area', 'mean_elevation', 'mean_slope', 
           'seed_triangle', 'p_fos_seed', 'volume', 
           'thickness', 'tsunami_potential_ratio']

In [ ]:
# Load lookuptable
lookup_table_path = os.path.join(volume_db.triangulation_dir, table_filename)
volume_db.logger.info(f"Load exceedance probabilities: {lookup_table_path}.")
diplacement_exceedance = np.load(lookup_table_path)
thresholds, exceedance_probs = diplacement_exceedance["thresholds"], diplacement_exceedance["probs"]

interpolator = interp1d(x=thresholds, y=exceedance_probs, fill_value=(1.,0.), bounds_error=True)

#volume_db.df[name] = interpolator(displacement_threshold)[volume_db.df.seed_triangle.to_numpy()]
probabilities = interpolator(displacement_threshold)


In [ ]:

volume_db.cursor.execute(f"SELECT id, seed_triangle FROM volumes")
rows = volume_db.cursor.fetchall()


In [ ]:
updates = [(float(probabilities[row[1]]), row[0]) for row in rows]

In [ ]:
updates

In [ ]:

# Check if the 'probability' column exists
if column_name not in COLUMN_NAMES:
    volume_db.cursor.execute(f"ALTER TABLE volumes ADD COLUMN {column_name} REAL")
    volume_db.logger.info("Added {column_name} column to the table.")

# Loop through each row, calculate the probability, and update the row
volume_db.cursor.executemany(f"UPDATE volumes SET {column_name} = ? WHERE id = ?", updates)

volume_db.conn.commit()

In [ ]:
[(row) for row in rows]

In [ ]:
# Get column names
column_names = [description[0] for description in volume_db.cursor.description]


In [ ]:
column_names

In [ ]:

# Check if the 'probability' column exists
if column_name not in column_names:
    volume_db.cursor.execute(f"ALTER TABLE volumes ADD COLUMN {column_name} REAL")
    volume_db.logger.info("Added 'probability' column to the table.")

# Loop through each row, calculate the probability, and update the row
volume_db.cursor.executemany(f"UPDATE volumes SET {column_name} = ? WHERE id = ?", (probabilities[row_dict["seed_triangle"]], row_dict['id']))

volume_db.conn.commit()